## Notebook to show how to read a 3DSG from the 3DSSG dataset, print information and plot 3D


In [18]:
import logging
import os
import sys
import torch
import json
from sentence_transformers import SentenceTransformer

os.environ["XDG_SESSION_TYPE"] = "x11"
os.environ["OPEN3D_DISABLE_WEB_VISUALIZER"] = "true"     

import numpy as np
import open3d as o3d

sys.path.append('../src')
from ssg import SceneGraph3D
from models import canonical_model_name, load_model, get_most_epochs_file
from utils import intersect_dict
from data import QueryData

%load_ext autoreload
%autoreload 2
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Constants definition


In [19]:
from consts import DATA_DIR, OUTPUT_DIR

SCENE_DIR = DATA_DIR / "scene_graphs"
OUT_MODEL_DIR = OUTPUT_DIR / "models"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SCAN_UUID = "02b33e03-be2b-2d54-9129-5d28efdd68fa"

## Load the 3DSSG dataset


## Instantiating the Graph


In [20]:
g = SceneGraph3D.from_json(SCENE_DIR / (SCAN_UUID + ".json"))
scan_id = g.scan_id
g.load_3d_data()

#### Plot the graph in 2D

In [21]:
SceneGraph3D.render(g, "../outputs/example_plot")  # renders to example_plot.pdf

#### Saving the graph as JSON (open and explore the JSON)


In [22]:
SceneGraph3D.to_json(g, "../outputs/example_output.json")

## Visualize 3D Object semantic segmented


In [23]:
vertex = g.pointcloud["vertex"]
xyz = np.vstack((vertex["x"], vertex["y"], vertex["z"])).T
colors = np.vstack((vertex["red"], vertex["green"], vertex["blue"])).T
obj_pcd = o3d.geometry.PointCloud()
obj_pcd.points = o3d.utility.Vector3dVector(xyz)
obj_pcd.colors = o3d.utility.Vector3dVector(colors / 255.0)
o3d.visualization.draw_plotly([obj_pcd])

### Visualize Mesh


In [24]:
o3d.visualization.draw([{"geometry": g.mesh, "name": "3RScan Mesh"}], show_ui=True)

### Load the Model

In [25]:
MODEL_NAME = canonical_model_name(
    "QueryInGat(in_dim=384,query_dim=384,hidden_dims=[64,64,64,64],edge_dim=384,out_dim=1,heads=10)"
)

model = load_model(MODEL_NAME)

most_trained = get_most_epochs_file(OUT_MODEL_DIR / MODEL_NAME)
model_state = torch.load(OUT_MODEL_DIR / MODEL_NAME / (str(most_trained) + ".pth"))
model.load_state_dict(model_state)

<All keys matched successfully>

In [26]:
SENTENCE_ENCODER = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)


def encode_node(node):
    node = intersect_dict({"ply_color", "label", "affordances", "attributes"}, node)
    node_desc = json.dumps(node)
    return torch.tensor(SENTENCE_ENCODER.encode([node_desc]).squeeze())


def encode_edge(edge):
    edge_desc = ", ".join(edge_d["name"] for edge_d in edge)
    return torch.tensor(SENTENCE_ENCODER.encode([edge_desc]).squeeze())


def encode_query(query):
    return torch.tensor(SENTENCE_ENCODER.encode([query]).squeeze())

2026-02-12 13:07:16,251 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-02-12 13:07:16,446 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-02-12 13:07:16,476 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-02-12 13:07:16,588 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-02-12 13:07:16,612 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-02-12 13:07:16,726 -

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-12 13:07:17,670 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-12 13:07:17,694 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-02-12 13:07:17,801 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-12 13:07:17,825 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolv

In [34]:
data, node_map, _ = SceneGraph3D.to_query_data(
    g, node_encoder=encode_node, edge_encoder=encode_edge, ret_node_maps=True
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [35]:
@torch.no_grad()
def run_model(model, node_map, query_data):
    model.eval()
    out = model(
        x=query_data.x,
        edge_index=query_data.edge_index,
        edge_attr=query_data.edge_attr,
        query=query_data.query.unsqueeze(0),
        batch=torch.zeros(query_data.num_nodes, dtype=torch.int32),
    )
    return [node_map[i] for i in torch.argsort(out)]

In [37]:
def query_model(model, node_map, query):
    query_data = QueryData(
        x=data.x,
        pos=data.pos,
        edge_index=data.edge_index,
        edge_attr=data.edge_attr,
        query=encode_query(query),
    )

    return run_model(model, node_map, query_data)

In [38]:
query_model(model, node_map, "Which object is a bed?")  # Expecting 4

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

['2',
 '1',
 '100',
 '29',
 '6',
 '9',
 '10',
 '11',
 '12',
 '21',
 '14',
 '20',
 '18',
 '8',
 '103',
 '15',
 '24',
 '4',
 '26',
 '25',
 '16',
 '13',
 '5',
 '3',
 '23',
 '27']

In [39]:
query_model(model, node_map, "Which object is a floor?")  # Expecting 1

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

['2',
 '27',
 '26',
 '23',
 '21',
 '29',
 '100',
 '12',
 '11',
 '10',
 '9',
 '8',
 '25',
 '6',
 '20',
 '1',
 '4',
 '13',
 '5',
 '3',
 '16',
 '24',
 '15',
 '18',
 '14',
 '103']

In [40]:
query_model(model, node_map, "Which object is elegant?")  # Expecting 14

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

['15',
 '103',
 '100',
 '29',
 '27',
 '26',
 '25',
 '24',
 '23',
 '20',
 '18',
 '16',
 '1',
 '14',
 '13',
 '12',
 '11',
 '10',
 '9',
 '8',
 '6',
 '5',
 '4',
 '3',
 '2',
 '21']

In [ ]:
query_model(model, node_map, "Which object is right of the stool?")  # Expecting 12

Batches:   0%|          | 0/1 [00:00<?, ?it/s]